# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gulzaibsharif-coder/flyrank-ml-capstone/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector uses page-level GSC/GA4 signals (impressions, position, sessions, engagement). CTR and clicks are deliberately excluded since they are used to define the target — see Section 3."

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

sample = dataset["train"][:5000]
df = pd.DataFrame(sample)

# Candidate features — things known independent of the click outcome itself
candidate_features = [
    "gsc_impressions",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
]

X = df[candidate_features].fillna(0)
print(X.shape)
X.head()


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

(5000, 8)


,gsc_impressions,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic
0,30,115,3.833333,0,0,0,0,0
1,5,358,71.600000,0,0,0,0,0
2,1,34,34.000000,0,0,0,0,0
3,6,140,23.333333,0,0,0,0,0
4,5,89,17.800000,0,0,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

All selected features are page-level, daily-aggregated numeric signals from Google Search Console (GSC) and Google Analytics 4 (GA4). None are categorical, so no encoding is required.

Each feature reflects search-performance or engagement activity that is recorded independently of whether a click occurred that day — meaning all are available *before* the prediction point, not derived from the outcome being predicted. Missing values are filled with 0 rather than dropped, to keep low-activity pages in the dataset. Missingness rates for each feature are printed in the code cell below; none is treated as informative on its own.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = {
    "gsc_impressions": "Search impressions for the page/day. Available pre-outcome.",
    "gsc_sum_position": "Sum of ranking positions across queries. Available pre-outcome.",
    "gsc_avg_position": "Average ranking position. Available pre-outcome.",
    "ga4_pageviews": "Pageviews in the reporting period. Available pre-outcome.",
    "ga4_sessions": "Sessions in the reporting period. Available pre-outcome.",
    "ga4_engaged_sessions": "Engaged sessions. Available pre-outcome.",
    "ga4_total_engagement_sec": "Total engagement time. Available pre-outcome.",
    "sessions_organic": "Organic-channel sessions. Available pre-outcome.",
}

for feat, note in feature_notes.items():
    missing_pct = df[feat].isna().mean() * 100
    print(f"{feat}: {note} | Missing: {missing_pct:.1f}%")

gsc_impressions: Search impressions for the page/day. Available pre-outcome. | Missing: 0.0%
gsc_sum_position: Sum of ranking positions across queries. Available pre-outcome. | Missing: 0.0%
gsc_avg_position: Average ranking position. Available pre-outcome. | Missing: 0.0%
ga4_pageviews: Pageviews in the reporting period. Available pre-outcome. | Missing: 0.0%
ga4_sessions: Sessions in the reporting period. Available pre-outcome. | Missing: 0.0%
ga4_engaged_sessions: Engaged sessions. Available pre-outcome. | Missing: 0.0%
ga4_total_engagement_sec: Total engagement time. Available pre-outcome. | Missing: 0.0%
sessions_organic: Organic-channel sessions. Available pre-outcome. | Missing: 0.0%


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

The target for this project is whether a page recorded at least one click (`gsc_clicks > 0`). Because of this, `gsc_clicks` itself — and any feature built from it, such as CTR (`gsc_clicks / gsc_impressions`) — is checked directly for correlation with the target.

An earlier version of the modeling workflow (Week 5–6) included `gsc_clicks` and CTR as features. This is a textbook target-leakage error: the target is literally defined from `gsc_clicks`, so any feature derived from that same column lets the model see the answer instead of predicting it. The correlation check below confirms this, and both fields are removed from the final feature set as a result.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the target the same way w06 does
target = (df["gsc_clicks"] > 0).astype(int)

# Attack: check correlation of EXCLUDED variables with the target
leak_candidates = ["gsc_clicks", "ctr" if "ctr" in df.columns else None]

print("Leakage hunt — correlation with target:")
print(f"gsc_clicks vs target correlation: {df['gsc_clicks'].corr(target):.3f}")

df["ctr_check"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)
print(f"ctr vs target correlation: {df['ctr_check'].corr(target):.3f}")

# Show why: target IS derived from gsc_clicks
print("\nTarget definition: target = (gsc_clicks > 0)")
print("Therefore gsc_clicks and any feature derived from it (e.g. ctr) directly encode the target.")


Leakage hunt — correlation with target:
gsc_clicks vs target correlation: 0.842
ctr vs target correlation: 0.591

Target definition: target = (gsc_clicks > 0)
Therefore gsc_clicks and any feature derived from it (e.g. ctr) directly encode the target.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Exclusions

The fields below were deliberately excluded from the feature vector, either because they leak the target or because they are identifiers with no predictive value and carry public-safety risk. Reasons for each are printed in the code cell below.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_features = {
    "gsc_clicks": "Used directly to construct the target (target = gsc_clicks > 0). Including it would leak the answer.",
    "ctr": "Derived from gsc_clicks / gsc_impressions — inherits the same leakage.",
    "client_hash_id": "Identifier, not a predictive signal; also a privacy/public-safety exclusion.",
    "content_hash_id": "Identifier, not a predictive signal; also a privacy/public-safety exclusion.",
}

print("Excluded features and reasons:")
for feat, reason in excluded_features.items():
    print(f"- {feat}: {reason}")

Excluded features and reasons:
- gsc_clicks: Used directly to construct the target (target = gsc_clicks > 0). Including it would leak the answer.
- ctr: Derived from gsc_clicks / gsc_impressions — inherits the same leakage.
- client_hash_id: Identifier, not a predictive signal; also a privacy/public-safety exclusion.
- content_hash_id: Identifier, not a predictive signal; also a privacy/public-safety exclusion.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.